In [12]:
#data splitting and maintaining directories. Done
# import os
# from sklearn.model_selection import train_test_split
# import shutil
# dir = "/kaggle/input/rnd-beg-dataset/test_buddha_yolo"
# content = os.listdir(dir)
# images = [x for x in content if ".jpg" in x]
# annotations = [x for x in content if ".txt" in x]
# test_images = "/kaggle/working/test_img_yolo"
# val_images = "/kaggle/working/val_img_yolo"
# os.makedirs(test_images, exist_ok=True)
# os.makedirs(val_images,exist_ok=True)
# train, test = train_test_split(images, random_state=104,test_size=0.25, shuffle=True)
# for img_path in train:
#     img_loc = os.path.join(test_images,img_path)
#     anno_loc = os.path.join(test_images,img_path[:-3]+"txt")
#     shutil.copyfile(os.path.join(dir,img_path), img_loc)
#     shutil.copyfile(os.path.join(dir,img_path[:-3]+"txt"), anno_loc)
# for img_path in test:
#     img_loc = os.path.join(val_images,img_path)
#     anno_loc = os.path.join(val_images,img_path[:-3]+"txt")
#     shutil.copyfile(os.path.join(dir,img_path), img_loc)
#     shutil.copyfile(os.path.join(dir,img_path[:-3]+"txt"), anno_loc)

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import os
from PIL import Image
import torch
import xml.etree.ElementTree as ET

In [ ]:
#extracting bounding box annotations from 

annotation_dir= "/kaggle/input/buddha-annotations-coco/"
dir_contents = os.listdir(annotation_dir)
anno_data = [annotation_dir + anno for anno in dir_contents]

In [ ]:
import re
def extract_number(xml_path):
    return int(re.search(r'\d+', xml_path.split("/")[-1]).group())
sorted_xml_paths = sorted(anno_data, key=extract_number)

sorted_xml_paths[:5]

In [ ]:
root = "/kaggle/input/d/lawliet07/rnd-beg-dataset/test_images/"
len(os.listdir(root))

In [ ]:
dataset_dir = os.listdir(root)
dataset_paths = [root + img for img in dataset_dir]
# dataset_paths[:5]

In [ ]:
def extract_img_number(img_path):
    return int(re.search(r'\d+', img_path.split('/')[-1]).group())

sorted_dataset_paths[:5]

# start here

In [ ]:
import os
import xml.etree.ElementTree as ET
from PIL import Image

# Define label map
label_map = {"broken": 0, "buddha": 1}  # YOLO starts with class IDs from 0

# Convert VOC-like XML annotations to YOLO format
def convert_to_yolo_format(annotation_dir, output_dir, image_dir):
    os.makedirs(output_dir, exist_ok=True)
    for annotation in os.listdir(annotation_dir):
        if annotation.endswith(".xml"):
            tree = ET.parse(os.path.join(annotation_dir, annotation))
            root = tree.getroot()
            image_filename = root.find("filename").text
            img_path = os.path.join(image_dir, image_filename)
            img = Image.open(img_path)
            width, height = img.size

            yolo_annotation_path = os.path.join(output_dir, os.path.splitext(annotation)[0] + ".txt")
            with open(yolo_annotation_path, "w") as yolo_file:
                for obj in root.findall("object"):
                    label = obj.find("name").text
                    bbox = obj.find("bndbox")
                    xmin = int(bbox.find("xmin").text)
                    ymin = int(bbox.find("ymin").text)
                    xmax = int(bbox.find("xmax").text)
                    ymax = int(bbox.find("ymax").text)

                    # Convert to YOLO format
                    x_center = (xmin + xmax) / (2 * width)
                    y_center = (ymin + ymax) / (2 * height)
                    box_width = (xmax - xmin) / width
                    box_height = (ymax - ymin) / height

                    # Write the annotation
                    class_id = label_map[label]
                    yolo_file.write(f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}\n")

# Paths
annotation_dir = "/kaggle/input/buddha-annotations-coco"
image_dir = "/kaggle/input/rnd-beg-dataset/test_images"
output_dir = "/kaggle/working/yolo_anno"
convert_to_yolo_format(annotation_dir, output_dir, image_dir)


In [ ]:
with open("/kaggle/working/dataset.yaml", "w") as f:
    f.write("train: /kaggle/input/rnd-beg-dataset/test_images\n")
    f.write("val: /kaggle/input/rnd-beg-dataset/test_images\n")
    f.write("nc: 2\n")  # Number of classes
    f.write("names: ['broken', 'buddha']\n")

In [ ]:
# Install YOLOv8 (if not already installed)
# pip install ultralytics

from ultralytics import YOLO

# Initialize a YOLOv8 model (pre-trained)
model = YOLO("yolov8s.pt")  # You can use yolov8n.pt, yolov8m.pt, etc.

# Train the model
model.train(data="/kaggle/working/dataset.yaml", epochs=15, imgsz=640, batch=8, device=0)

# Save the model
model.save("yolo_buddha_model.pt")


In [ ]:
# Perform inference on test images
test_image_dir = "/kaggle/input/rnd-beg-dataset/test_images"
results = model.predict(source=test_image_dir, save=True, conf=0.25)

# Visualize predictions
results[0].show()
